# İstatistiksel Analiz: Çoklu Karşılaştırma Düzeltmesi ve Güven Aralıkları

> Açıklamalar Türkçedir; tablolar ve sonuç dosyaları İngilizce hazırlanmıştır.

**Amaç.** `00_cicids2017_robustness_experiments.ipynb` (CIC-IDS2017) ve `01_csecicids2018_replication_comparison.ipynb` (CSE-CIC-IDS2018) seed bazındaki ham sonuçları kaydeder. Bu notebook model eğitmez; yalnızca bu kayıtlı sonuçları okuyarak iki ek istatistiksel analiz yapar:

1. **Holm düzeltmesi.** H1, her veri setinde 18 kombinasyon (3 model × 2 saldırgan × 3 özellik grubu) için Recall ve F1 ayrı ayrı sınanarak toplam 36 Spearman testiyle değerlendirilir. Bu kadar çok test yapıldığında bazı testlerin tesadüfen anlamlı çıkma olasılığı artar. Holm yöntemi p-değerlerini test sayısına göre düzelterek bu riski kontrol eder.
2. **%95 güven aralıkları.** Ana karşılaştırmalar (temiz F1 ve referans senaryodaki göreli F1 düşüşü) için 3 seed üzerinden t dağılımıyla %95 güven aralığı hesaplanır. Yalnızca 3 seed olduğundan aralıklar geniştir; bu, sonuçların ne kadar kesin olduğunu dürüstçe gösterir.

Çalıştırmadan önce `00_cicids2017_robustness_experiments.ipynb` ve `01_csecicids2018_replication_comparison.ipynb` çalıştırılmış olmalıdır. Çalışma süresi birkaç saniyedir.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr, t as t_dist

METRICS_DIR = Path("../results/metrics")
OUT_DIR = METRICS_DIR / "statistics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "CIC-IDS2017": METRICS_DIR / "raw_experiment_B_per_seed.csv",
    "CSE-CIC-IDS2018": METRICS_DIR / "2018" / "raw_experiment_B_per_seed.csv",
}
MODEL_NAMES = ["Random Forest", "XGBoost", "MLP"]
EPS_REF = 0.50
ALPHA = 0.05

TABLE_STYLES = [
    {"selector": "caption", "props": "caption-side: top; text-align: left; font-weight: 600; "
                                     "font-size: 13px; color: #0b0b0b; padding: 4px 0 8px 0;"},
    {"selector": "th", "props": "background: #f3f2ee; color: #0b0b0b; font-weight: 600; text-align: left; "
                                "padding: 6px 12px; border-bottom: 1px solid #c3c2b7;"},
    {"selector": "td", "props": "padding: 5px 12px; border-bottom: 1px solid #ecebe6; "
                                "font-variant-numeric: tabular-nums;"},
]


def show_table(df, title, filename=None, precision=4):
    # Tabloyu CSV olarak kaydeder ve notebook'ta biçimlendirilmiş hâlde gösterir.
    if filename:
        df.to_csv(OUT_DIR / filename, index=False, encoding="utf-8-sig")
    styler = (df.style.set_caption(title).hide(axis="index")
              .set_table_styles(TABLE_STYLES).format(precision=precision, thousands=","))
    display(styler)


results = {ds: pd.read_csv(path) for ds, path in SOURCES.items()}
for ds, df in results.items():
    print(f"{ds:<16} {len(df):>4} satır · seedler {sorted(df['Seed'].unique())}")

CIC-IDS2017       324 satır · seedler [np.int64(42), np.int64(100), np.int64(2024)]
CSE-CIC-IDS2018   324 satır · seedler [np.int64(42), np.int64(100), np.int64(2024)]


## 1. H1 için Holm düzeltmesi

Her kombinasyonda ε ile metrik arasındaki Spearman korelasyonu 18 nokta üzerinden hesaplanır (6 ε seviyesi × 3 seed). H1'in bir kombinasyonda desteklenmesi için hem Recall hem F1 korelasyonunun negatif ve anlamlı olması gerekir. Holm düzeltmesi her veri setindeki 36 teste ayrı ayrı uygulanır: p-değerleri küçükten büyüğe sıralanır, *k*. sıradaki değer (36 − *k* + 1) ile çarpılır ve sıralama monoton kalacak şekilde düzeltilir.

In [2]:
def holm(p_values):
    # Holm–Bonferroni düzeltmesi: düzeltilmiş p-değerlerini orijinal sırayla döndürür.
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, i in enumerate(np.argsort(p)):
        running_max = max(running_max, min(1.0, (m - rank) * p[i]))
        adjusted[i] = running_max
    return adjusted


h1_rows, summary_rows = [], []
for ds, df in results.items():
    rows = []
    for (attacker, group, name), d in df.groupby(["Attacker", "Group", "Model"], sort=False):
        row = {"Dataset": ds, "Attacker": attacker, "Group": group, "Model": name}
        for metric in ["Recall", "F1"]:
            rho, p = spearmanr(d["Epsilon"], d[metric])
            row[f"ρ ({metric})"] = rho
            row[f"p ({metric})"] = 1.0 if np.isnan(p) else p
        rows.append(row)
    t = pd.DataFrame(rows)

    adjusted = holm(np.r_[t["p (Recall)"], t["p (F1)"]])
    t["p Holm (Recall)"], t["p Holm (F1)"] = adjusted[:len(t)], adjusted[len(t):]
    negative = (t["ρ (Recall)"] < 0) & (t["ρ (F1)"] < 0)
    raw_ok = negative & (t["p (Recall)"] < ALPHA) & (t["p (F1)"] < ALPHA)
    holm_ok = negative & (t["p Holm (Recall)"] < ALPHA) & (t["p Holm (F1)"] < ALPHA)
    t["H1 (uncorrected)"] = np.where(raw_ok, "Yes", "No")
    t["H1 (Holm)"] = np.where(holm_ok, "Yes", "No")
    h1_rows.append(t)
    summary_rows.append({"Dataset": ds, "Tests": 2 * len(t),
                         "Supported (uncorrected)": f"{raw_ok.sum()} / {len(t)}",
                         "Supported (Holm)": f"{holm_ok.sum()} / {len(t)}",
                         "Lost after correction": ", ".join(
                             f"{r.Model} ({r.Attacker}, {r.Group})" for r in t[raw_ok & ~holm_ok].itertuples()) or "—"})

h1_table = pd.concat(h1_rows, ignore_index=True)
show_table(pd.DataFrame(summary_rows),
           f"Table S1 — H1 with Holm correction (36 Spearman tests per dataset, α = {ALPHA})",
           "h1_holm_summary.csv")
show_table(h1_table, "Table S2 — Spearman tests per combination: raw and Holm-adjusted p-values",
           "h1_holm_per_combination.csv", precision=4)

Dataset,Tests,Supported (uncorrected),Supported (Holm),Lost after correction
CIC-IDS2017,36,17 / 18,14 / 18,"MLP (Random, Size), MLP (Random, Size+Time), MLP (Adaptive, Time)"
CSE-CIC-IDS2018,36,17 / 18,16 / 18,"XGBoost (Random, Time)"


Dataset,Attacker,Group,Model,ρ (Recall),p (Recall),ρ (F1),p (F1),p Holm (Recall),p Holm (F1),H1 (uncorrected),H1 (Holm)
CIC-IDS2017,Random,Size,Random Forest,-0.8371,0.0000,-0.8371,0.0000,0.0002,0.0002,Yes,Yes
CIC-IDS2017,Random,Time,Random Forest,-0.9813,0.0000,-0.9813,0.0000,0.0000,0.0000,Yes,Yes
CIC-IDS2017,Random,Size+Time,Random Forest,-0.9875,0.0000,-0.9875,0.0000,0.0000,0.0000,Yes,Yes
CIC-IDS2017,Adaptive,Size,Random Forest,-0.9248,0.0000,-0.9248,0.0000,0.0000,0.0000,Yes,Yes
CIC-IDS2017,Adaptive,Time,Random Forest,-0.9875,0.0000,-0.9875,0.0000,0.0000,0.0000,Yes,Yes
CIC-IDS2017,Adaptive,Size+Time,Random Forest,-0.9875,0.0000,-0.9875,0.0000,0.0000,0.0000,Yes,Yes
CIC-IDS2017,Random,Size,XGBoost,-0.8980,0.0000,-0.8980,0.0000,0.0000,0.0000,Yes,Yes
CIC-IDS2017,Random,Time,XGBoost,-0.8379,0.0000,-0.8783,0.0000,0.0002,0.0000,Yes,Yes
CIC-IDS2017,Random,Size+Time,XGBoost,-0.9750,0.0000,-0.9750,0.0000,0.0000,0.0000,Yes,Yes
CIC-IDS2017,Adaptive,Size,XGBoost,-0.9321,0.0000,-0.9258,0.0000,0.0000,0.0000,Yes,Yes


## 2. Ana karşılaştırmalar için %95 güven aralıkları

Her model ve veri seti için seed bazındaki değerlerden ortalama, standart sapma ve t dağılımına dayalı %95 güven aralığı hesaplanır: ortalama ± t₀.₉₇₅,ₙ₋₁ · s / √n. n = 3 olduğunda t₀.₉₇₅,₂ ≈ 4.30'dur; bu yüzden aralıklar standart sapmanın yaklaşık 2.5 katı genişliğindedir.

Hesaplanan büyüklükler:
- **Temiz F1** (ε = 0),
- **Göreli F1 düşüşü (%)**: uyarlamalı saldırgan, Boyut+Zaman, ε = `EPS_REF`,
- **Kaçırılan saldırı artışı (yüzde puan)**: aynı senaryoda, test kümesindeki saldırılara oranla.

In [3]:
def ci95(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean, std = values.mean(), values.std(ddof=1)
    half = t_dist.ppf(0.975, n - 1) * std / np.sqrt(n)
    return mean, std, mean - half, mean + half


ci_rows = []
for ds, df in results.items():
    ref = df[(df["Attacker"] == "Adaptive") & (df["Group"] == "Size+Time")]
    clean = ref[ref["Epsilon"] == 0].set_index(["Seed", "Model"])
    attacked = ref[ref["Epsilon"] == EPS_REF].set_index(["Seed", "Model"])
    missed = lambda d: d["FN"] / (d["FN"] + d["TP"]) * 100
    per_seed = pd.DataFrame({
        "Clean F1": clean["F1"],
        "Relative F1 drop (%)": (1 - attacked["F1"] / clean["F1"]) * 100,
        "Missed attack increase (pp)": missed(attacked) - missed(clean),
    }).reset_index()
    for name in MODEL_NAMES:
        d = per_seed[per_seed["Model"] == name]
        for quantity in ["Clean F1", "Relative F1 drop (%)", "Missed attack increase (pp)"]:
            mean, std, lo, hi = ci95(d[quantity])
            ci_rows.append({"Dataset": ds, "Model": name, "Quantity": quantity, "n seeds": len(d),
                            "Mean": mean, "Std": std, "95% CI low": lo, "95% CI high": hi})

ci_table = pd.DataFrame(ci_rows)
show_table(ci_table,
           f"Table S3 — 95% confidence intervals over seeds (t distribution; adaptive attacker, Size+Time, ε = {EPS_REF * 100:.0f}%)",
           "ci95_main_comparison.csv", precision=4)

Dataset,Model,Quantity,n seeds,Mean,Std,95% CI low,95% CI high
CIC-IDS2017,Random Forest,Clean F1,3,0.9971,0.0002,0.9966,0.9976
CIC-IDS2017,Random Forest,Relative F1 drop (%),3,4.0726,1.8200,-0.4487,8.5938
CIC-IDS2017,Random Forest,Missed attack increase (pp),3,7.7349,3.3744,-0.6476,16.1174
CIC-IDS2017,XGBoost,Clean F1,3,0.9980,0.0001,0.9977,0.9983
CIC-IDS2017,XGBoost,Relative F1 drop (%),3,1.0373,0.0978,0.7943,1.2804
CIC-IDS2017,XGBoost,Missed attack increase (pp),3,2.0445,0.1912,1.5695,2.5194
CIC-IDS2017,MLP,Clean F1,3,0.9870,0.0014,0.9836,0.9904
CIC-IDS2017,MLP,Relative F1 drop (%),3,0.4097,0.1387,0.0651,0.7543
CIC-IDS2017,MLP,Missed attack increase (pp),3,0.7931,0.2682,0.1268,1.4594
CSE-CIC-IDS2018,Random Forest,Clean F1,3,0.9233,0.0008,0.9214,0.9252


## 3. Göreli F1 düşüşü aralıklarının karşılaştırılması

Güven aralıkları çakışmıyorsa iki modelin göreli kaybı arasındaki farkın seed değişkenliğinden büyük olduğu söylenebilir. Çakışma, farkın anlamsız olduğunu kesin olarak göstermez; ancak o karşılaştırmanın 3 seed ile ayırt edilemediği anlamına gelir.

In [4]:
drop = ci_table[ci_table["Quantity"] == "Relative F1 drop (%)"].set_index(["Dataset", "Model"])
pair_rows = []
for ds in SOURCES:
    for i, a in enumerate(MODEL_NAMES):
        for b in MODEL_NAMES[i + 1:]:
            ra, rb = drop.loc[(ds, a)], drop.loc[(ds, b)]
            overlap = not (ra["95% CI high"] < rb["95% CI low"] or rb["95% CI high"] < ra["95% CI low"])
            pair_rows.append({"Dataset": ds, "Comparison": f"{a} vs {b}",
                              "Drop A (%)": ra["Mean"], "Drop B (%)": rb["Mean"],
                              "CIs overlap?": "Yes" if overlap else "No"})
show_table(pd.DataFrame(pair_rows), "Table S4 — Do the 95% CIs of the relative F1 drop overlap?",
           "ci95_pairwise_overlap.csv", precision=2)

Dataset,Comparison,Drop A (%),Drop B (%),CIs overlap?
CIC-IDS2017,Random Forest vs XGBoost,4.07,1.04,Yes
CIC-IDS2017,Random Forest vs MLP,4.07,0.41,Yes
CIC-IDS2017,XGBoost vs MLP,1.04,0.41,No
CSE-CIC-IDS2018,Random Forest vs XGBoost,7.40,0.66,No
CSE-CIC-IDS2018,Random Forest vs MLP,7.40,18.97,No
CSE-CIC-IDS2018,XGBoost vs MLP,0.66,18.97,No


### Yorumlar

**1. Holm düzeltmesi H1'in genel sonucunu değiştirmiyor, ancak MLP için temkinli olmayı gerektiriyor.**
- **CIC-IDS2017:** H1'i destekleyen kombinasyon sayısı 17'den 14'e iner. Desteklenmeyen dört kombinasyonun hepsi MLP'ye aittir. Rastgele saldırganla zaman grubu düzeltmeden önce de desteklenmiyordu. Diğer üçü (rastgele boyut, rastgele boyut+zaman, uyarlamalı zaman) düzeltmeyle desteğini kaybeder; bunlarda Recall'daki düşüş anlamlı kalır, F1'deki düşüş anlamlılığını yitirir. Ağaç modellerinin 12 kombinasyonunun tamamı düzeltmeden sonra da H1'i destekler.
- **CSE-CIC-IDS2018:** Sayı 17'den 16'ya iner. Desteğini kaybeden tek kombinasyon rastgele saldırganla zaman pertürbasyonundaki XGBoost'tur; bu senaryoda XGBoost'un kaybı zaten çok küçüktür.

Sonuç olarak H1, özellikle ağaç tabanlı modeller ve uyarlamalı saldırgan için çoklu karşılaştırma düzeltmesinden sonra da güçlü biçimde desteklenmektedir.

**2. Güven aralıkları 2017'deki sıralamanın ne kadar kesin olduğunu gösteriyor.**
- 2017'de Random Forest'ın göreli F1 düşüşü seedler arasında belirgin biçimde değişir (seed 42'de %2.0, diğer iki seed'de %5.0 ve %5.3). Bu nedenle %95 güven aralığı geniştir ([−0.45, 8.59]) ve sıfırı içerir.
- Tablo S4'e göre 2017'de yalnızca XGBoost ile MLP arasındaki farkın aralıkları çakışmaz. Random Forest'ın aralığı diğer iki modelinkiyle çakışır; yani 3 seed ile Random Forest'ın diğer modellerden farkı istatistiksel olarak ayırt edilemez.
- 2018'de üç modelin aralıkları birbirinden tamamen ayrıktır; XGBoost > Random Forest > MLP sıralaması nettir.

**Sonuç.** `00_cicids2017_robustness_experiments.ipynb` ve `01_csecicids2018_replication_comparison.ipynb`'deki ana sonuçlar bu ek analizlerle de geçerlidir. Ancak 2017'de Random Forest'ın sıralamadaki yeri seed'e duyarlıdır; bu, raporda sınırlılık olarak belirtilmiştir. Yalnızca 3 seed kullanıldığı için güven aralıkları geniştir; daha kesin bir sıralama için daha fazla seed gerekir.